In [2]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score


In [3]:
# Generate 1_D data
np.random.seed(42)
X = np.random.uniform(-3, 3, (100, 1))
# True slope is 2.5, true intercept is 5.0
y = 2.5 + X.ravel() + 5.0 + np.random.normal(0, 3.5, 100)

In [4]:
# Standard linear regression(OLS)
lin_reg = LinearRegression()
lin_reg.fit(X, y)

y_pred_lin = lin_reg.predict(X)
r2_lin = r2_score(y, y_pred_lin)

print("Standard Linear Regression (OLS)")
print(f"R2 Score: {r2_lin:.6f}")
print(f"Slope (m): {lin_reg.coef_[0]:.6f}")
print(f"Intercept: {lin_reg.intercept_:.6f}")

Standard Linear Regression (OLS)
R2 Score: 0.145995
Slope (m): 0.731799
Intercept: 7.448233


In [9]:
# Custom ridge 1D class
class CustomRidge1D:
  def __init__(self, alpha=1.0):
    self.alpha = float(alpha)
    self.slope_ = None
    self.intercept_ = None

  def fit(self, X, y):
    x_vec = X.ravel()
    y_vec = y.ravel()

    # center the data
    x_mean = np.mean(x_vec)
    y_mean = np.mean(y_vec)
    x_centered = x_vec - x_mean
    y_centered = y_vec - y_mean

    # ridge closed-form 1D Equation: m = sum(x*y) / (sum(x^2) + alpha)
    numerator = np.sum(x_centered * y_centered)
    denominator = np.sum(x_centered ** 2) + self.alpha

    self.slope_ = numerator / denominator
    self.intercept_ = y_mean - (self.slope_ * x_mean)
    return self

  def predict(self, X):
    return (X.ravel() * self.slope_) + self.intercept_

In [10]:
alpha_value = 10.0  # High penalty to force shrinkage
ridge_1d = CustomRidge1D(alpha=alpha_value).fit(X, y)

y_pred_ridge = ridge_1d.predict(X)
r2_ridge = r2_score(y, y_pred_ridge)

print(f"--- 2. CUSTOM RIDGE REGRESSION (Alpha = {alpha_value}) ---")
print(f"R2 Score:  {r2_ridge:.6f}")
print(f"Slope (m): {ridge_1d.slope_:.6f}")
print(f"Intercept: {ridge_1d.intercept_:.6f}\n")

--- 2. CUSTOM RIDGE REGRESSION (Alpha = 10.0) ---
R2 Score:  0.145857
Slope (m): 0.709311
Intercept: 7.444210



In [11]:
print("--- 3. THE ENGINEERING TAKEAWAY ---")
print(f"Slope Shrinkage: {lin_reg.coef_[0] - ridge_1d.slope_:.6f} (Ridge slope is flatter)")
print(f"R2 Drop:         {r2_lin - r2_ridge:.6f} (Ridge intentionally worsens training score)")

--- 3. THE ENGINEERING TAKEAWAY ---
Slope Shrinkage: 0.022488 (Ridge slope is flatter)
R2 Drop:         0.000138 (Ridge intentionally worsens training score)


# Analyzing the Output
When you run this code, you are going to see something that seems completely counter-intuitive at first: Ridge Regression has a worse $R^2$ score than Linear Regression.

1. This is not a bug; it is the entire point of the Bias-Variance Tradeoff.The Slope Drops: Look at the slope values. Standard OLS calculates the slope purely to chase the data points. Ridge's $\alpha$ penalty dragged the slope closer to zero. The line literally became flatter.

2. The Intercept Stays Stable: Notice how the intercept barely changed. Because we centered the data in our custom class before calculating the math, we successfully protected the intercept from being penalized.

3. The $R^2$ Drops: Because Ridge forcibly flattened the line away from the absolute "perfect" training fit, the training $R^2$ score goes down. Ridge is intentionally sacrificing training accuracy (introducing Bias) to prevent the model from being overly sensitive to outliers (reducing Variance).